# eRisk Notebook

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import sys
import time
from datetime import datetime
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from app.cli import run_eval
from core.llm import get_llm_usage
from core.runtime_policy import (
    auto_backend_switch_enabled,
    cuda_runtime,
    min_cuda_vram_gb,
    resolve_detector_backend,
    resolve_persona_backend,
)

OUTPUTS = Path('outputs')
OUTPUTS.mkdir(exist_ok=True)

Project root: /home/mdel2424/dev/eRisk_Honours


In [2]:
def set_env(**kwargs):
    for key, value in kwargs.items():
        if value is None:
            os.environ.pop(key, None)
        else:
            os.environ[key] = str(value)

def backend_snapshot():
    cuda_ok, vram = cuda_runtime()
    return {
        'auto_backend_switch': auto_backend_switch_enabled(),
        'cuda_available': cuda_ok,
        'cuda_vram_gb': round(vram, 2),
        'min_cuda_vram_gb': min_cuda_vram_gb(),
        'resolved_detector_backend': resolve_detector_backend(),
        'resolved_persona_backend': resolve_persona_backend(),
    }

backend_snapshot()


{'auto_backend_switch': True,
 'cuda_available': False,
 'cuda_vram_gb': 0.0,
 'min_cuda_vram_gb': 8.0,
 'resolved_detector_backend': 'openrouter',
 'resolved_persona_backend': 'openrouter_sim'}

In [3]:
def run_quick_eval(
    personas: int = 8,
    seed: int = 42,
    eval_mode: str = 'mixed_holdout',
    prompt_version: str = 'v1',
    save_diagnostics: bool = True,
    max_api_calls: int = 120,
    trace_level: str = 'compact',
    fit_calibrator: str = 'auto',
):
    started = time.time()
    run_eval(
        persona_count=personas,
        seed=seed,
        eval_mode=eval_mode,
        prompt_version=prompt_version,
        save_diagnostics=save_diagnostics,
        max_api_calls=max_api_calls,
        trace_level=trace_level,
        fit_calibrator_policy=fit_calibrator,
    )
    elapsed = time.time() - started
    print(f'Elapsed: {elapsed:.1f}s')
    return elapsed


In [ ]:
quick_cfg = {
    'personas': 8,
    'seed': 42,
    'eval_mode': 'mixed_holdout',
    'prompt_version': os.getenv('PROMPT_VERSION', 'v1'),
    'save_diagnostics': True,
    'max_api_calls': 180,
    'trace_level': 'compact',
    'fit_calibrator': 'auto',
}
quick_cfg


{'personas': 8,
 'seed': 42,
 'eval_mode': 'mixed_holdout',
 'prompt_version': 'v1',
 'save_diagnostics': True,
 'max_api_calls': 120,
 'trace_level': 'compact',
 'fit_calibrator': 'auto'}

In [5]:
# Run one eval iteration (same engine as CLI)
run_quick_eval(**quick_cfg)


--- Eval Mode: mixed_holdout | personas=8 | seed=42 | prompts=v1 ---
Backend info: auto_switch=on | cuda_available=False | vram_gb=0.00 | min_vram_gb=8.00 | cuda_gate=fail
Resolved backends: detector=openrouter [meta-llama/llama-3-8b-instruct] | persona=openrouter_sim [meta-llama/llama-3-8b-instruct]
Runtime controls: trace_level=compact | max_api_calls=120
Calibrator policy: requested=auto | enabled=False | min_train_records=10

=== Persona synth-3 (synthetic) ===
Evaluation calls=23/120 [####--------------------] 1/6
=== Persona synth-6 (synthetic) ===
Evaluation calls=43/120 [########----------------] 2/6
=== Persona synth-1 (synthetic) ===
Evaluation calls=65/120 [############------------] 3/6
=== Persona synth-2 (synthetic) ===
Evaluation calls=85/120 [################--------] 4/6
=== Persona 1 (official) ===
Evaluation calls=105/120 [####################----] 5/6
=== Persona 2 (official) ===
Evaluation calls=120/120 [########################] 6/6

Stopping eval early: API call b

230.83426356315613

In [6]:
def load_json(path: str | Path):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(p)
    return json.loads(p.read_text(encoding='utf-8'))

def metric_headline(metrics: dict) -> dict:
    return {
        'primary_eval_split': metrics.get('primary_eval_split'),
        'primary_metrics': metrics.get('primary_metrics', {}),
        'llm_usage': metrics.get('llm_usage', {}),
    }


In [7]:
metrics = load_json(OUTPUTS / 'metrics_run_local.json')
failure = load_json(OUTPUTS / 'failure_report_run_local.json')
results = load_json(OUTPUTS / 'results_run_local.json')

metric_headline(metrics)


{'primary_eval_split': 'overall_labeled',
 'primary_metrics': {'binary_accuracy': 0.75,
  'binary_f1': 0.8,
  'bdi_mae': 9.75,
  'symptom_f1_at_4': 0.1964,
  'avg_turns_to_decision': 10.0,
  'risk_recall': 0.0,
  'objective': 0.65},
 'llm_usage': {'calls_total': 120,
  'prompt_tokens_total': 57056,
  'completion_tokens_total': 6752,
  'total_tokens_total': 63808,
  'errors_total': 0,
  'max_calls': 120,
  'calls_remaining': 0}}

In [8]:
{
    'failure_counters': failure.get('failure_counters', {}),
    'route_distribution': failure.get('route_distribution', {}),
    'evidence_nonempty_rate': failure.get('evidence_nonempty_rate'),
    'avg_evidence_per_turn': failure.get('avg_evidence_per_turn'),
    'runtime_llm_usage': get_llm_usage(),
}


{'failure_counters': {'extract_empty': 23,
  'extract_json_parse_fail': 17,
  'budget_exceeded': 1},
 'route_distribution': {'cognitive': 41, 'somatic': 17},
 'evidence_nonempty_rate': 0.5862,
 'avg_evidence_per_turn': 0.7069,
 'runtime_llm_usage': {'calls_total': 120,
  'prompt_tokens_total': 57056,
  'completion_tokens_total': 6752,
  'total_tokens_total': 63808,
  'errors_total': 0,
  'max_calls': 120,
  'calls_remaining': 0}}

In [9]:
results[:6]


[{'LLM': 'synth-3',
  'bdi-score': 9,
  'key-symptoms': ['Sadness',
   'Past Failure',
   'Loss of Energy',
   'Changes in Sleeping Pattern']},
 {'LLM': 'synth-6',
  'bdi-score': 9,
  'key-symptoms': ['Sadness',
   'Loss of Pleasure',
   'Worthlessness',
   'Loss of Energy']},
 {'LLM': 'synth-1',
  'bdi-score': 6,
  'key-symptoms': ['Guilty Feelings', 'Loss of Pleasure', 'Sadness']},
 {'LLM': 'synth-2',
  'bdi-score': 9,
  'key-symptoms': ['Loss of Pleasure',
   'Sadness',
   'Past Failure',
   'Loss of Energy']},
 {'LLM': '1',
  'bdi-score': 12,
  'key-symptoms': ['Loss of Pleasure',
   'Changes in Sleeping Pattern',
   'Sadness',
   'Agitation']},
 {'LLM': '2', 'bdi-score': 2, 'key-symptoms': ['Sadness']}]

In [10]:
diagnostics = load_json(OUTPUTS / 'diagnostics_run_local.json')

def flatten_timeline(diag_payload: list[dict], limit: int | None = None) -> list[dict]:
    rows = []
    for rec in diag_payload:
        persona_id = rec.get('LLM')
        for turn in rec.get('timeline', []):
            route_decision = turn.get('route_decision') or {}
            stop_decision = turn.get('stop_decision') or {}
            rows.append({
                'LLM': persona_id,
                'turn': turn.get('turn'),
                'route': route_decision.get('chosen_node'),
                'policy': route_decision.get('policy'),
                'evidence_count': len(turn.get('latest_evidence', []) or []),
                'pred_label': turn.get('predicted_label'),
                'pred_bdi': turn.get('predicted_bdi_score'),
                'confidence': turn.get('global_confidence'),
                'stop': stop_decision.get('should_stop'),
                'stop_reason': stop_decision.get('reason'),
            })
    if limit is not None:
        return rows[:limit]
    return rows

rows = flatten_timeline(diagnostics, limit=40)
rows


[{'LLM': 'synth-3',
  'turn': 1,
  'route': 'cognitive',
  'policy': 'info_gain',
  'evidence_count': 0,
  'pred_label': 'control',
  'pred_bdi': 0,
  'confidence': 0.8807970779778824,
  'stop': False,
  'stop_reason': 'continue'},
 {'LLM': 'synth-3',
  'turn': 2,
  'route': 'somatic',
  'policy': 'lexical',
  'evidence_count': 1,
  'pred_label': 'control',
  'pred_bdi': 2,
  'confidence': 0.7651050946177169,
  'stop': False,
  'stop_reason': 'continue'},
 {'LLM': 'synth-3',
  'turn': 3,
  'route': 'somatic',
  'policy': 'lexical',
  'evidence_count': 0,
  'pred_label': 'control',
  'pred_bdi': 2,
  'confidence': 0.84102507635801,
  'stop': False,
  'stop_reason': 'continue'},
 {'LLM': 'synth-3',
  'turn': 4,
  'route': 'cognitive',
  'policy': 'lexical',
  'evidence_count': 0,
  'pred_label': 'control',
  'pred_bdi': 2,
  'confidence': 0.84102507635801,
  'stop': False,
  'stop_reason': 'continue'},
 {'LLM': 'synth-3',
  'turn': 5,
  'route': 'somatic',
  'policy': 'lexical',
  'evide

In [11]:
def run_sweep(configs: list[dict], label: str = 'sweep'):
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    sweep_dir = OUTPUTS / f'{label}_{stamp}'
    sweep_dir.mkdir(parents=True, exist_ok=True)

    summaries = []
    for idx, cfg in enumerate(configs, start=1):
        print(f'\n=== Sweep run {idx}/{len(configs)}: {cfg}')
        run_quick_eval(**cfg)

        run_dir = sweep_dir / f'run_{idx:02d}'
        run_dir.mkdir(parents=True, exist_ok=True)
        for name in [
            'interactions_run_local.json',
            'results_run_local.json',
            'metrics_run_local.json',
            'failure_report_run_local.json',
            'config_used.json',
            'diagnostics_run_local.json',
        ]:
            src = OUTPUTS / name
            if src.exists():
                shutil.copy2(src, run_dir / name)

        metrics = load_json(run_dir / 'metrics_run_local.json')
        primary = metrics.get('primary_metrics', {})
        summaries.append({
            'run': idx,
            'cfg': cfg,
            'primary_split': metrics.get('primary_eval_split'),
            'binary_f1': primary.get('binary_f1'),
            'bdi_mae': primary.get('bdi_mae'),
            'avg_turns_to_decision': primary.get('avg_turns_to_decision'),
        })

    return sweep_dir, summaries


In [ ]:
# # Example sweep (run multiple configs)
# sweep_cfgs = [
#     {**quick_cfg, 'max_api_calls': 100},
#     {**quick_cfg, 'max_api_calls': 140},
# ]
# sweep_dir, sweep_summary = run_sweep(sweep_cfgs, label='fast_eval')
# sweep_dir, sweep_summary



=== Sweep run 1/2: {'personas': 8, 'seed': 42, 'eval_mode': 'mixed_holdout', 'prompt_version': 'v1', 'save_diagnostics': True, 'max_api_calls': 100, 'trace_level': 'compact', 'fit_calibrator': 'auto'}
--- Eval Mode: mixed_holdout | personas=8 | seed=42 | prompts=v1 ---
Backend info: auto_switch=on | cuda_available=False | vram_gb=0.00 | min_vram_gb=8.00 | cuda_gate=fail
Resolved backends: detector=openrouter [meta-llama/llama-3-8b-instruct] | persona=openrouter_sim [meta-llama/llama-3-8b-instruct]
Runtime controls: trace_level=compact | max_api_calls=100
Calibrator policy: requested=auto | enabled=False | min_train_records=10

=== Persona synth-3 (synthetic) ===
Evaluation calls=21/100 [####--------------------] 1/6
=== Persona synth-6 (synthetic) ===
Evaluation calls=41/100 [########----------------] 2/6
=== Persona synth-1 (synthetic) ===
Evaluation calls=63/100 [############------------] 3/6
=== Persona synth-2 (synthetic) ===
Evaluation calls=83/100 [################--------] 4/6


(PosixPath('outputs/fast_eval_20260219_225354'),
 [{'run': 1,
   'cfg': {'personas': 8,
    'seed': 42,
    'eval_mode': 'mixed_holdout',
    'prompt_version': 'v1',
    'save_diagnostics': True,
    'max_api_calls': 100,
    'trace_level': 'compact',
    'fit_calibrator': 'auto'},
   'primary_split': 'overall_labeled',
   'binary_f1': 0.8,
   'bdi_mae': 9.5,
   'avg_turns_to_decision': 10.0},
  {'run': 2,
   'cfg': {'personas': 8,
    'seed': 42,
    'eval_mode': 'mixed_holdout',
    'prompt_version': 'v1',
    'save_diagnostics': True,
    'max_api_calls': 140,
    'trace_level': 'compact',
    'fit_calibrator': 'auto'},
   'primary_split': 'overall_labeled',
   'binary_f1': 0.8,
   'bdi_mae': 8.75,
   'avg_turns_to_decision': 10.0}])